[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-03-config-tag-decorators.ipynb#scrollTo=bb000001)

---
# Day 3 · Decorators Part 1: @config, @tag, @extract_columns
**certified-journeys / hamilton-certified** · Day 3 · Decorators

> **Goal for today:** Use `@config.when` to branch on environment, `@tag` to attach metadata to nodes, and `@extract_columns` to split a DataFrame into individually-named Series.

In [ ]:
%pip install -q sf-hamilton

## Step 1 · @config.when — Environment Branching

`@config.when` lets you define **alternative implementations of the same node** that activate based on a config flag — no if/else inside your functions.

```python
from hamilton.function_modifiers import config

@config.when(data_source='csv')
def raw_data__csv(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

@config.when(data_source='parquet')
def raw_data__parquet(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)
```

The `__` suffix in the function name is a naming convention: `node_name__variant`. Hamilton strips the suffix and both functions produce a node named `raw_data`. Only one activates based on the config.

| Variant | Activated when |
|---|---|
| `@config.when(k=v)` | Config has key k with value v |
| `@config.when_not(k=v)` | Config does NOT have k=v |
| `@config.when_in(k=[v1,v2])` | Config k is one of v1, v2 |
| `@config.when_not_in(k=[v1])` | Config k is NOT in the list |

In [ ]:
import sys, types, io
import pandas as pd
from hamilton import driver
from hamilton.function_modifiers import config, tag, extract_columns

# Two implementations of the same node: customer_data
@config.when(env='dev')
def customer_data__dev(n_rows: int) -> pd.DataFrame:
    """Dev: generate synthetic data."""
    import numpy as np
    rng = np.random.default_rng(42)
    return pd.DataFrame({
        'age':    rng.integers(18, 80, n_rows).astype(float),
        'spend':  rng.exponential(50, n_rows),
        'tenure': rng.integers(0, 60, n_rows).astype(float),
    })

@config.when(env='prod')
def customer_data__prod(csv_path: str) -> pd.DataFrame:
    """Prod: read from CSV file."""
    return pd.read_csv(csv_path)

def age(customer_data: pd.DataFrame) -> pd.Series:
    return customer_data['age']

def spend(customer_data: pd.DataFrame) -> pd.Series:
    return customer_data['spend']

# Build module and driver with dev config
cfg_module = types.ModuleType('cfg_module')
for fn in [customer_data__dev, customer_data__prod, age, spend]:
    setattr(cfg_module, fn.__name__, fn)
sys.modules['cfg_module'] = cfg_module

dr_dev = driver.Builder().with_modules(cfg_module).with_config({'env': 'dev'}).build()
result = dr_dev.execute(['age', 'spend'], inputs={'n_rows': 5})
print('Dev result:')
print('age:', result['age'].values)
print('spend:', result['spend'].round(2).values)

### What just happened?
- **`@config.when(env='dev')`** activates `customer_data__dev`; `customer_data__prod` is excluded from the graph entirely.
- Switching to prod is a **Driver config change, not a code change** — just pass `{'env': 'prod'}` and the other branch activates.
- **No if/else** inside functions — cleaner, individually testable branches.

## Step 2 · @tag — Node Metadata

`@tag` attaches arbitrary key-value metadata to a node. Common uses:
- Mark feature type (`numerical`, `categorical`, `boolean`)
- Mark owning team
- Mark PII sensitivity
- Filter nodes at execute time

```python
from hamilton.function_modifiers import tag

@tag(feature_type='numerical', owner='data-team', pii='false')
def age_normalized(age: pd.Series, age_mean: float, age_std: float) -> pd.Series:
    return (age - age_mean) / age_std
```

Tags are queryable via the graph: `dr.list_available_variables()` returns `NodeMetadata` which includes tags.

In [ ]:
@tag(feature_type='numerical', pii='false')
def age_mean(age: pd.Series) -> float:
    return float(age.mean())

@tag(feature_type='numerical', pii='false')
def age_std(age: pd.Series) -> float:
    return float(age.std())

@tag(feature_type='numerical', pii='false')
def age_normalized(age: pd.Series, age_mean: float, age_std: float) -> pd.Series:
    return (age - age_mean) / age_std

@tag(feature_type='boolean', pii='false')
def is_high_spender(spend: pd.Series) -> pd.Series:
    return spend > spend.quantile(0.75)

# Add tagged functions to the module
for fn in [age_mean, age_std, age_normalized, is_high_spender]:
    setattr(cfg_module, fn.__name__, fn)

dr_tagged = driver.Builder().with_modules(cfg_module).with_config({'env': 'dev'}).build()

# Query tags from the graph
print('=== Nodes with feature_type tag ===')
for v in dr_tagged.list_available_variables():
    tags = v.tags
    if 'feature_type' in tags:
        print(f'  {v.name}: feature_type={tags["feature_type"]}')

### What just happened?
- **`@tag(key=value)`** attaches metadata without changing function behavior.
- **Tags are queryable** via `v.tags` — build feature registries, lineage docs, or selective execution from tags.
- **Multiple tags** are just keyword arguments to the decorator — no limit.

## Step 3 · @extract_columns — DataFrame → Named Series

`@extract_columns` takes a function that returns a DataFrame and exposes each column as a **named Series node**.

```python
from hamilton.function_modifiers import extract_columns

@extract_columns('col_a', 'col_b', 'col_c')
def raw_features(customer_data: pd.DataFrame) -> pd.DataFrame:
    return customer_data[['col_a', 'col_b', 'col_c']]
```

This creates nodes `col_a`, `col_b`, and `col_c` that downstream functions can depend on individually — instead of every downstream function having to pull a column from a DataFrame.

**When to use:** When you have a source that naturally returns a DataFrame (CSV, SQL query, API) and you want to expose individual columns as first-class DAG nodes.

In [ ]:
from hamilton.function_modifiers import extract_columns

@extract_columns('age', 'spend', 'tenure')
def raw_features(customer_data: pd.DataFrame) -> pd.DataFrame:
    """Extract the three raw feature columns from the source DataFrame."""
    return customer_data[['age', 'spend', 'tenure']]

@tag(feature_type='numerical')
def tenure_years(tenure: pd.Series) -> pd.Series:
    """Convert tenure from months to years."""
    return tenure / 12.0

extract_module = types.ModuleType('extract_module')
for fn in [customer_data__dev, raw_features, tenure_years, is_high_spender]:
    setattr(extract_module, fn.__name__, fn)
sys.modules['extract_module'] = extract_module

dr_ext = driver.Builder().with_modules(extract_module).with_config({'env': 'dev'}).build()

result = dr_ext.execute(['age', 'spend', 'tenure', 'tenure_years', 'is_high_spender'],
                        inputs={'n_rows': 6})
print('tenure (months):', result['tenure'].values.round(1))
print('tenure_years:   ', result['tenure_years'].values.round(2))
print('is_high_spender:', result['is_high_spender'].values)

### What just happened?
- **`@extract_columns('age', 'spend', 'tenure')`** auto-creates three downstream nodes from one function's DataFrame output.
- **`tenure_years`** depends on the `tenure` node — not on `raw_features` directly. Hamilton wires it transparently.
- This is the recommended pattern for ingesting tabular data: one loader function + `@extract_columns` to give each column a proper DAG identity.

## Step 4 · Combining All Three Decorators

Decorators compose — a function can have `@config.when`, `@tag`, and `@extract_columns` together. The order matters: `@extract_columns` must be outermost when combined with `@tag`.

In [ ]:
# Real-world pattern: config-gated loader + tagged features
from hamilton.function_modifiers import config, tag

@config.when(env='dev')
@tag(source='synthetic', pii='false')
def enriched_customer_data__dev(n_rows: int) -> pd.DataFrame:
    """Dev: synthetic data with extra derived columns already computed."""
    import numpy as np
    rng = np.random.default_rng(0)
    df = pd.DataFrame({
        'age':    rng.integers(18, 80, n_rows).astype(float),
        'spend':  rng.exponential(50, n_rows),
        'tenure': rng.integers(0, 60, n_rows).astype(float),
    })
    df['clv'] = df['spend'] * df['tenure']  # customer lifetime value proxy
    return df

@tag(feature_type='numerical', owner='ml-team')
def clv(enriched_customer_data: pd.DataFrame) -> pd.Series:
    """Customer lifetime value from enriched data."""
    return enriched_customer_data['clv']

combo_module = types.ModuleType('combo_module')
for fn in [enriched_customer_data__dev, clv]:
    setattr(combo_module, fn.__name__, fn)
sys.modules['combo_module'] = combo_module

dr_combo = driver.Builder().with_modules(combo_module).with_config({'env': 'dev'}).build()
res = dr_combo.execute(['clv'], inputs={'n_rows': 4})
print('CLV values:', res['clv'].round(2).values)

# Inspect tags on the clv node
for v in dr_combo.list_available_variables():
    if v.name == 'clv':
        print('clv tags:', v.tags)

### What just happened?
- **Decorators compose cleanly** — `@config.when` controls which implementation is active; `@tag` adds metadata.
- The underlying function logic is unchanged — decorators are purely declarative.
- Tags survive graph construction and are visible on the resulting node metadata.

In [ ]:
# Challenge: add a @config.when(env='test') variant of enriched_customer_data
# that returns a hardcoded 3-row DataFrame (deterministic for tests)
# Then build a Driver with env='test' and verify it uses your test data

# @config.when(env='test')
# def enriched_customer_data__test() -> pd.DataFrame:
#     ...

print('Add the test variant and rebuild the driver with env=test!')

---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| `@config.when(k=v)` | Only one variant of a node activates per Driver config |
| `__suffix` convention | `node_name__variant` — Hamilton strips the suffix |
| `@tag(k=v)` | Attaches queryable metadata without affecting execution |
| `@extract_columns` | Turns one DataFrame-returning function into N named Series nodes |
| Decorator composition | Stack decorators; `@extract_columns` must be outermost with `@tag` |

> **Tip:** `@config.when` is how Hamilton handles environment branching without if/else clutter. Name your configs like feature flags — e.g. `config.when(data_source="bigquery")`.

---
## What's next
**Day 4** → Build your first real feature engineering pipeline: 8+ features from a tabular dataset, tagged by type, collected into a feature DataFrame.

Mark Day 3 complete in your [tracker](../index.html).